# Task 1.1 — Data Preparation and Validation Pipeline
**Unsupervised Learning Project 2025/2026 — Hotel Booking Demand**

## RQ1 — Segmentation Question

**Question:** Can we identify distinct booking behaviour profiles among hotel guests, based only on information available at the time of booking, that are meaningful for hotel revenue management and operations?

**Unit of analysis:** Each row in the dataset represents one booking record.

Clustering is appropriate here because there are no predefined guest segments — the goal is to discover latent structure in booking patterns (lead time, stay duration, guest composition, channel, etc.) without relying on outcome labels.

## Data Documentation

- **Source:** Course release v1 — `hotel_bookings_course_release_v1.csv`
- **Original:** Kaggle — [jessemostipak/hotel-booking-demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand/data)
- **Reference:** António, N., de Almeida, A., & Nunes, L. (2019). Hotel booking demand datasets. *Data in Brief*, 22, 41–49. https://doi.org/10.1016/j.dib.2018.11.126
- **License:** CC0 1.0 Public Domain (as stated in the course release)
- **What each row represents:** One hotel booking record (one stay or cancellation)
- **Time span:** Arrivals between July 2015 and August 2017, across two hotels (City Hotel and Resort Hotel)
- **Known data quality issues:**
  - `children`: 4 NaN values
  - `country`: ~0.4% missing values
  - `agent` / `company`: NULL encoded as the string `"NULL"` (not a proper NaN)
  - Some rows have 0 adults + 0 children + 0 babies (impossible — data entry error)
  - `adr` has at least one negative value (data entry error)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SEED = 12345
np.random.seed(SEED)

df_raw = pd.read_csv('../hotel_bookings_course_release_v1.csv')
print('Shape:', df_raw.shape)
df_raw.head(3)

## Leakage Control & Feature Exclusions

Clustering inputs must only use variables available **at booking time**. The following are excluded:

| Variable | Reason |
|---|---|
| `is_canceled` | Outcome variable — unknown at booking time |
| `reservation_status` | Post-event status — realised after the stay |
| `reservation_status_date` | Date of post-event status — post-arrival |
| `agent` | High-cardinality ID-like field (hundreds of numeric codes); no meaningful encoding |
| `company` | High-cardinality ID-like field; >90% NULL |

The outcome variables (`is_canceled`, `reservation_status`) are retained separately for **post-hoc profiling only**.

In [ ]:
EXCLUDED = ['is_canceled', 'reservation_status', 'reservation_status_date', 'agent', 'company']

# Keep outcome vars for post-hoc profiling
df_labels = df_raw[['is_canceled', 'reservation_status']].copy()

df = df_raw.drop(columns=EXCLUDED).copy()
print('Shape after exclusions:', df.shape)

## Data Quality Fixes

In [ ]:
# Fix 1: children NaN -> 0 (no child is the correct interpretation for missing)
df['children'] = df['children'].fillna(0).astype(int)

# Fix 2: remove rows with 0 adults + 0 children + 0 babies (impossible)
zero_guests = (df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)
print(f'Zero-guest rows removed: {zero_guests.sum()}')
df = df[~zero_guests].copy()

# Fix 3: negative ADR -> NaN (will be median-imputed in pipeline)
neg_adr = df['adr'] < 0
print(f'Negative ADR values set to NaN: {neg_adr.sum()}')
df.loc[neg_adr, 'adr'] = np.nan

# Align profiling labels
df_labels = df_labels.loc[df.index]

print('Working dataset shape:', df.shape)

## Feature Set Definition

In [ ]:
NUMERICAL = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
    'total_of_special_requests'
]

CATEGORICAL = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]

print(f'Numerical features  ({len(NUMERICAL)}): {NUMERICAL}')
print(f'Categorical features ({len(CATEGORICAL)}): {CATEGORICAL}')

## Missingness Report

In [ ]:
df_features = df[NUMERICAL + CATEGORICAL].copy()

print('=== Missingness — Numerical features ===')
miss_num = df_features[NUMERICAL].isnull().sum()
miss_num_pct = (miss_num / len(df_features) * 100).round(3)
miss_num_df = pd.DataFrame({'n_missing': miss_num, 'pct_missing': miss_num_pct})
print(miss_num_df[miss_num_df['n_missing'] > 0].to_string())
print('(All others: 0 missing)' if (miss_num == 0).all() else '')

print()
print('=== Missingness — Categorical features ===')
miss_cat = df_features[CATEGORICAL].isnull().sum()
miss_cat_pct = (miss_cat / len(df_features) * 100).round(3)
miss_cat_df = pd.DataFrame({'n_missing': miss_cat, 'pct_missing': miss_cat_pct})
print(miss_cat_df[miss_cat_df['n_missing'] > 0].to_string())
if (miss_cat == 0).all():
    print('(All others: 0 missing)')

## Outlier Report (Numerical — IQR method)

In [ ]:
print('=== Outliers — Numerical features (IQR rule: outside Q1-1.5*IQR / Q3+1.5*IQR) ===')
rows = []
for col in NUMERICAL:
    s = df_features[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((s < lo) | (s > hi)).sum()
    rows.append({'feature': col, 'lower_fence': round(lo,2), 'upper_fence': round(hi,2),
                 'n_outliers': n_out, 'pct': round(n_out/len(s)*100, 2)})

outlier_df = pd.DataFrame(rows).set_index('feature')
print(outlier_df.to_string())

print()
print('=== Outliers — Categorical features ===')
print('Not applicable (no numerical range). Rare categories handled via frequency threshold in pipeline.')

## Preprocessing Pipeline

**Numerical:** median imputation (robust to outliers/skew) → StandardScaler (z-score normalization)

**Categorical:** mode imputation → rare categories (< 1% frequency) grouped into `"Other"` → OneHotEncoder with `handle_unknown='ignore'` (unseen categories at inference map to all-zero vector)

**Implied distance/metric:** After this pipeline the representation is a dense numeric matrix with standardised numerical columns and binary OHE columns. The natural and implied distance is **Euclidean distance in this standardised mixed space**, which is compatible with k-means and Ward hierarchical clustering.

In [ ]:
# Group rare categories (< 1% frequency) into 'Other' before OHE
MIN_FREQ = 0.01
for col in CATEGORICAL:
    freq = df_features[col].value_counts(normalize=True)
    rare = freq[freq < MIN_FREQ].index
    if len(rare) > 0:
        df_features[col] = df_features[col].apply(lambda x: 'Other' if x in rare else x)
        print(f'{col}: {len(rare)} rare categories grouped into "Other"')

In [ ]:
numerical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,  NUMERICAL),
    ('cat', categorical_pipeline, CATEGORICAL),
], remainder='drop')

X = preprocessor.fit_transform(df_features)

ohe_names = (
    preprocessor
    .named_transformers_['cat']
    .named_steps['encode']
    .get_feature_names_out(CATEGORICAL)
    .tolist()
)
feature_names = NUMERICAL + ohe_names
X_df = pd.DataFrame(X, columns=feature_names, index=df_features.index)

print(f'Final matrix shape : {X_df.shape}')
print(f'  - Numerical dims : {len(NUMERICAL)}')
print(f'  - OHE dims       : {len(ohe_names)}')
print(f'  - Total samples  : {X_df.shape[0]:,}')
print(f'NaN in output      : {X_df.isnull().sum().sum()}')

## Final Feature Set Summary

| | |
|---|---|
| **Samples** | see output above |
| **Numerical features** | 17 (standardised, median-imputed) |
| **Categorical features** | 10 (one-hot encoded, rare→Other) |
| **Total dimensions after OHE** | see output above |
| **Implied distance** | Euclidean in standardised mixed space |
| **Compatible algorithms** | k-means, MiniBatch k-means, Ward hierarchical |
| **Excluded (leakage)** | `is_canceled`, `reservation_status`, `reservation_status_date` |
| **Excluded (ID-like)** | `agent`, `company` |
| **Retained for profiling only** | `is_canceled`, `reservation_status` |